# Active Space & Symmetry Reduction

Qubit reduction by Z2 symmetry -- the same idea as active-space selection for larger molecules.

**Objectives:**
- Diagonalize the full four-qubit H2 Hamiltonian for its ground energy
- Build the Z2-tapered single-qubit H2 Hamiltonian and confirm the same ground state
- See the qubit count collapse from 4 to 1 without changing the physics
- Connect this to active-space selection for larger molecules

**Reference:** See [`../GUIDE.md`](../GUIDE.md) for concept explanations and context.

<!-- browser-runnable -->

In [ ]:
# Setup: Run this cell first
from braket.circuits import Circuit
from braket.devices import LocalSimulator
import numpy as np
import matplotlib.pyplot as plt
from lib.utils.statevector import statevector
np.random.seed(0)
device = LocalSimulator()

In [ ]:
# --- Chemistry kit (auto-provided; real STO-3G H2 data + exact helpers) ---
# H2 Jordan-Wigner qubit Hamiltonian at R = 0.75 Angstrom (STO-3G, big-endian:
# character k of each Pauli string acts on qubit k, qubit 0 = leftmost = MSB).
# Coefficients are REAL precomputed values (PennyLane differentiable Hartree-Fock).
H2_TERMS = [
    ("IIII", -0.1097305),
    ("IIIZ", -0.21886309),
    ("IIZI", -0.21886309),
    ("IIZZ", 0.17395379),
    ("IZII", 0.16988453),
    ("IZIZ", 0.12005143),
    ("IZZI", 0.16549432),
    ("XXYY", -0.04544288),
    ("XYYX", 0.04544288),
    ("YXXY", 0.04544288),
    ("YYXX", -0.04544288),
    ("ZIII", 0.16988453),
    ("ZIIZ", 0.16549432),
    ("ZIZI", 0.12005143),
    ("ZZII", 0.16821199),
]
H2_FCI = -1.137117   # exact ground-state energy (lowest eigenvalue), Hartree
H2_HF = -1.116151    # Hartree-Fock energy <1100|H|1100>, Hartree
# Z2 symmetry-tapered single-qubit form: H = c0*I + cz*Z + cx*X (ground = c0 - hypot(cz, cx))
H2_C0, H2_CZ, H2_CX = -0.338656, 0.777495, 0.181772

_PAULI = {
    "I": np.array([[1, 0], [0, 1]], dtype=complex),
    "X": np.array([[0, 1], [1, 0]], dtype=complex),
    "Y": np.array([[0, -1j], [1j, 0]], dtype=complex),
    "Z": np.array([[1, 0], [0, -1]], dtype=complex),
}

def pauli_matrix(pauli_string):
    """Dense matrix of a big-endian Pauli string (qubit 0 = leftmost factor)."""
    m = np.array([[1.0 + 0j]])
    for ch in pauli_string:
        m = np.kron(m, _PAULI[ch])
    return m

def hamiltonian_matrix(terms):
    """Dense Hamiltonian sum(coeff * Pauli) from a list of (pauli_string, coeff)."""
    n = len(terms[0][0])
    h = np.zeros((2 ** n, 2 ** n), dtype=complex)
    for s, c in terms:
        h += c * pauli_matrix(s)
    return h

def hamiltonian_energy(state_vector, terms):
    """Expectation <psi|H|psi> for a qcsim state vector (real, in Hartree)."""
    h = hamiltonian_matrix(terms)
    return float(np.real(np.vdot(state_vector, h @ state_vector)))

def h2_double_excitation(theta):
    """HF |1100> plus the single Givens double excitation |1100> <-> |0011>.
    At the optimal theta this ansatz reaches the exact H2 ground state."""
    c = Circuit()
    c.x(0); c.x(1)
    c.cnot(2, 3); c.cnot(0, 2); c.h(0); c.h(3); c.cnot(0, 1); c.cnot(2, 3)
    c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(0, 3); c.h(3); c.cnot(3, 1); c.ry(0, -theta / 8); c.ry(1, theta / 8)
    c.cnot(2, 1); c.cnot(2, 0); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(3, 1); c.h(3); c.cnot(0, 3); c.ry(0, theta / 8); c.ry(1, -theta / 8)
    c.cnot(0, 1); c.cnot(2, 0); c.h(0); c.h(3); c.cnot(0, 2); c.cnot(2, 3)
    return c
# --- end chemistry kit ---

## 1. The full four-qubit H2 Hamiltonian

`H2_TERMS` is the exact STO-3G H2 problem (R = 0.75 A): fifteen big-endian Pauli strings on **four qubits**. Its ground energy is the lowest eigenvalue; the kit gives `hamiltonian_matrix` and `H2_FCI` = -1.137117 Ha.

In [ ]:
H_full = hamiltonian_matrix(H2_TERMS)
n_qubits_full = len(H2_TERMS[0][0])
E_full_ground = np.linalg.eigvalsh(H_full)[0]
print(f"{len(H2_TERMS)} terms, {n_qubits_full} qubits | lowest {E_full_ground:.6f} | FCI {H2_FCI:.6f}")
assert np.isclose(E_full_ground, H2_FCI, atol=1e-4)
print("OK: four-qubit Hamiltonian gives the exact FCI ground energy.")

## 2. The same molecule on a single qubit

Each conserved Z2 symmetry tapers away one qubit: `H_tapered = c0*I + cz*Z + cx*X` (kit `H2_C0`, `H2_CZ`, `H2_CX`). Eigenvalues `c0 +/- sqrt(cz^2 + cx^2)`, ground energy `c0 - hypot(cz, cx)`.

In [ ]:
I2 = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex)
Z2 = np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
X2 = np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
H_tapered = H2_C0 * I2 + H2_CZ * Z2 + H2_CX * X2
n_qubits_tapered = 1
E_tapered_ground = np.linalg.eigvalsh(H_tapered)[0]
E_closed_form = H2_C0 - np.hypot(H2_CZ, H2_CX)
print(f"lowest {E_tapered_ground:.6f} | c0-hypot {E_closed_form:.6f} | FCI {H2_FCI:.6f}")
assert np.isclose(E_tapered_ground, E_closed_form, atol=1e-12)
assert np.isclose(E_tapered_ground, H2_FCI, atol=1e-4)
print("OK: one-qubit tapered Hamiltonian gives the exact FCI ground energy.")

## 3. Same ground state, a quarter of the qubits

The four-qubit and one-qubit operators agree on the lowest eigenvalue and both equal `H2_FCI`. Only the qubit budget changed.

In [ ]:
assert np.isclose(E_full_ground, E_tapered_ground, atol=1e-4)
qubit_reduction = n_qubits_full - n_qubits_tapered
print(f"qubits {n_qubits_full} -> {n_qubits_tapered} ({qubit_reduction} removed) | dim {2**n_qubits_full} -> {2**n_qubits_tapered}")
print("OK: 4 -> 1 qubit reduction, ground state preserved.")

## 4. Reaching the ground state

The one-qubit ground state comes from `eigh`; the four-qubit one is prepared via the kit's `h2_double_excitation(theta)`, reading energy from `statevector(...)` through `hamiltonian_energy`. The state vector is big-endian and spans the full four-qubit register.

In [ ]:
vecs = np.linalg.eigh(H_tapered)[1]
E_from_vec = np.real(vecs[:, 0].conj() @ H_tapered @ vecs[:, 0])
assert np.isclose(E_from_vec, H2_FCI, atol=1e-4)
thetas = np.linspace(-np.pi, np.pi, 401)
energies = np.array([hamiltonian_energy(statevector(h2_double_excitation(t)), H2_TERMS) for t in thetas])
best = int(np.argmin(energies))
print(f"best theta {thetas[best]:+.4f} -> {energies[best]:.6f} Ha (FCI {H2_FCI:.6f})")
assert np.isclose(energies[best], H2_FCI, atol=1e-3)
sv = statevector(h2_double_excitation(thetas[best]))
print(f"state vector length = {len(sv)} (= 2**4)")
print("OK: both representations reach the same -1.137 Ha ground state.")

## 5. Scaling up: this is active-space selection

Active-space selection drops uncorrelated orbitals: cheap **Hartree-Fock**, **keep orbitals near the Fermi level**, **freeze the rest**. The classical validator is **PySCF CASCI** (`mcscf.CASCI(mf, ncas=4, nelecas=4).kernel()` -- not run here, PySCF is browser-unavailable). H2O in STO-3G is a **14-qubit** problem cut to **4-to-8 qubits**. The naive qubit count is almost never the real one.

## 6. Exercises

Four exercises to push the tapering idea further: the excited state on one
qubit, checking the tapered spectrum against the full one, the mean-field
energy after tapering, and a single-qubit VQE. Each gives you a prompt with two
tiered hints -- open only what you need -- a scaffold cell to write your attempt
in, and a check cell that tells you when you have it. They reuse only the kit
names (`H_tapered`, `H_full`, `H2_C0`, `H2_CZ`, `H2_CX`, `H2_FCI`, `H2_HF`) and
the allowed imports.

### Exercise 1 — The tapered excited state

`H_tapered = c0*I + cz*Z + cx*X` has exactly two eigenvalues. Section 2 read the
lower one (the ground energy). Now read the upper one -- H2's first excited
energy in this one-qubit picture.

Define `E_tapered_excited`: the higher of the tapered Hamiltonian's two
eigenvalues. Compare it to `E_tapered_ground`; the two straddle the identity
coefficient `H2_C0`.

<details><summary>Hint 1 — nudge</summary>

Section 2 gave the ground energy in closed form as `c0 - hypot(cz, cx)`. A real
2x2 symmetric matrix has exactly two eigenvalues, placed symmetrically about the
midpoint of its trace. What is the OTHER one?

</details>
<details><summary>Hint 2 — approach</summary>

Two routes. Algebraically, take the closed form from Hint 1 and flip the sign
of its square-root (`np.hypot`) term -- the excited energy sits the same
distance ABOVE the trace midpoint as the ground energy sits below it.
Numerically, diagonalize `H_tapered` with `np.linalg.eigvalsh` and read the
LAST (largest) eigenvalue instead of the first.

</details>

In [ ]:
# Exercise 1: Read the excited-state energy of the one-qubit tapered Hamiltonian.
# Define: E_tapered_excited -- the higher of H_tapered's two eigenvalues.

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    assert E_tapered_excited > E_tapered_ground, (
        "the excited state is the HIGHER of the two eigenvalues"
    )
    assert np.isclose(E_tapered_excited, float(np.linalg.eigvalsh(H_tapered)[-1]), atol=1e-9), (
        "E_tapered_excited should be the largest eigenvalue of the tapered Hamiltonian"
    )
    assert np.isclose(E_tapered_excited + E_tapered_ground, 2 * H2_C0, atol=1e-9), (
        "the two eigenvalues sit symmetrically about the identity coefficient"
    )

### Exercise 2 — Spectrum overlap

Tapering is supposed to preserve physics, not just the ground energy. Check that
BOTH tapered eigenvalues live inside the full four-qubit spectrum.

Define `full_spectrum`: all `2**4` eigenvalues of `H_full` (via
`np.linalg.eigvalsh`). Then define `matched_distances`: for each of the two
tapered eigenvalues, the distance to its NEAREST value in `full_spectrum`. If
tapering is faithful, both distances collapse to nearly zero.

<details><summary>Hint 1 — nudge</summary>

`eigvalsh` returns every eigenvalue, not only the lowest. The tapered operator
describes the same molecule restricted to one symmetry sector, so its two
eigenvalues should each coincide with an eigenvalue of the big operator.

</details>
<details><summary>Hint 2 — approach</summary>

Compute `full_spectrum = np.linalg.eigvalsh(H_full)` (sixteen values) and
`np.linalg.eigvalsh(H_tapered)` (two values). For each tapered value, take
`np.min(np.abs(full_spectrum - value))` and collect the two results; the larger
of the two should land far below `1e-3`.

</details>

In [ ]:
# Exercise 2: Confirm both tapered eigenvalues appear in the four-qubit spectrum.
# Define: full_spectrum -- all 2**4 eigenvalues of H_full; matched_distances --
# the nearest-neighbour distance in full_spectrum for each tapered eigenvalue.

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    assert len(full_spectrum) == 2 ** n_qubits_full, (
        "the full spectrum should hold all 2**4 eigenvalues"
    )
    assert len(matched_distances) == 2, (
        "one distance per tapered eigenvalue (ground and excited)"
    )
    assert max(matched_distances) < 1e-3, (
        "each tapered eigenvalue should reappear in the four-qubit spectrum -- "
        "tapering drops qubits from this symmetry sector, not its eigenvalues"
    )

### Exercise 3 — Tapered Hartree-Fock vs FCI

The ground state of `H_tapered` is a superposition -- the `X` term tilts it off
axis. What energy does a plain computational basis state give? That is the
mean-field (Hartree-Fock) estimate in this one-qubit picture.

Define `basis_energies`: a dict mapping `"0"` and `"1"` to the expectation
`<b|H_tapered|b>` of each single-qubit basis state. Then define `hf_tapered`:
the LOWER of those two energies. Confirm it sits above `H2_FCI` -- and notice
which known number it matches.

<details><summary>Hint 1 — nudge</summary>

A computational basis state is not an eigenstate of `H_tapered` -- the
off-diagonal `X` term is exactly the correlation it cannot see. So its energy (a
diagonal expectation) must land above the true ground energy, by the variational
principle.

</details>
<details><summary>Hint 2 — approach</summary>

Build each basis vector as a length-2 complex array with a single `1.0` entry
(`[1, 0]` for `"0"`, `[0, 1]` for `"1"`), then evaluate `v.conj() @ H_tapered @
v`. Store both under their bit labels, take the smaller as `hf_tapered`, and
compare it against `H2_HF` and `H2_FCI`.

</details>

In [ ]:
# Exercise 3: Score the single-qubit basis states against the tapered Hamiltonian.
# Define: basis_energies -- dict {"0": <0|H_tapered|0>, "1": <1|H_tapered|1>};
# hf_tapered -- the lower of the two energies (the mean-field pick).

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    assert set(basis_energies) == {"0", "1"}, (
        "score both single-qubit computational basis states"
    )
    assert all(abs(np.imag(e)) < 1e-9 for e in basis_energies.values()), (
        "a computational-basis expectation <b|H|b> is real"
    )
    assert np.isclose(hf_tapered, min(np.real(e) for e in basis_energies.values()), atol=1e-9), (
        "hf_tapered is the LOWER of the two basis-state energies -- the mean-field choice"
    )
    assert hf_tapered > H2_FCI + 1e-4, (
        "one computational basis state cannot beat the correlated ground state: HF sits above FCI"
    )
    assert np.isclose(hf_tapered, H2_HF, atol=1e-4), (
        "the best tapered basis state reproduces the Hartree-Fock energy"
    )

### Exercise 4 — Single-qubit Ry VQE

Section 4 ran a full VQE on four qubits. After tapering, the whole search
collapses to ONE angle: sweep `Ry(theta)|0>` on a single qubit and find the
rotation that minimizes the energy against `H_tapered`.

Define `ry_thetas`: a grid of angles spanning at least `[-pi, pi]`. Define
`ry_energies`: the energy `<psi|H_tapered|psi>` of `psi = Ry(theta)|0>` at each
angle, in the same order. Define `E_vqe`: the minimum of `ry_energies`. Confirm
it reaches `H2_FCI` -- one rotation is all it takes.

<details><summary>Hint 1 — nudge</summary>

`Ry(theta)|0>` traces every real single-qubit state as `theta` runs through a
full turn. `H_tapered` has no `Y` term, so its ground state is real -- somewhere
on that sweep you pass exactly through it. The variational principle keeps every
sampled energy at or above `H2_FCI`.

</details>
<details><summary>Hint 2 — approach</summary>

Make a grid with `np.linspace(-np.pi, np.pi, 401)`. For each angle build the
state with `statevector(Circuit().ry(0, theta))`, then compute `np.real(psi.conj()
@ H_tapered @ psi)`. Collect the energies and take `min(...)` for `E_vqe` -- the
same read-energy-from-a-state-vector move as Section 4, but on one qubit.

</details>

In [ ]:
# Exercise 4: Minimize a single-qubit Ry(theta) state against the tapered Hamiltonian.
# Define: ry_thetas -- grid of angles over [-pi, pi]; ry_energies -- energy of
# Ry(theta)|0> at each angle; E_vqe -- the minimum energy found.

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    assert len(ry_energies) == len(ry_thetas), "record one energy per theta on your scan"
    assert np.isclose(E_vqe, min(ry_energies), atol=1e-9), "E_vqe is the lowest energy you found"
    assert E_vqe >= H2_FCI - 1e-6, (
        "the variational principle floors every Ry(theta) state at H2_FCI"
    )
    assert np.isclose(E_vqe, H2_FCI, atol=1e-3), (
        "a single Ry rotation on the tapered qubit is enough to reach the exact ground energy"
    )

## Summary

- `H2_TERMS`: fifteen Pauli strings on **four qubits**, lowest eigenvalue the exact `H2_FCI` = -1.137 Ha.
- Z2-tapered it is `H = c0*I + cz*Z + cx*X` on **one qubit**; ground energy `c0 - hypot(cz, cx)` = `H2_FCI`.
- Both describe the identical ground state: **4 qubits collapse to 1**, physics untouched.
- Active-space selection in miniature; PySCF CASCI validates the choice and turns H2O from 14 qubits into 4-to-8.

**Next:** [`07-excited-states.ipynb`](07-excited-states.ipynb) -- subspace-search VQE for H2's first excited state.